# nf-core/sarek Sandwich Wrapper

**Genesis Workbench** — Zero-Fork Germline & Somatic Variant Calling Orchestration

This notebook implements the **sandwich wrapper pattern** for `nf-core/sarek` (variant calling). The pipeline is a completely untouched black box — all Eisai-specific logic lives in the Databricks-native bread layers.

```
┌─────────────────────────────────────────────────────┐
│             PRE-FLIGHT (Databricks-native)           │
│  ┌───────────┐ ┌──────────┐ ┌────────────────────┐  │
│  │ Validate  │ │ Build    │ │ Generate           │  │
│  │ Inputs &  │ │ Sarek    │ │ eisai_sarek.config │  │
│  │ Env       │ │ Sample-  │ │ (process selectors)│  │
│  │           │ │ sheet    │ │                    │  │
│  └───────────┘ └──────────┘ └────────────────────┘  │
├─────────────────────────────────────────────────────┤
│         nf-core/sarek (UNTOUCHED BLACK BOX)         │
│                                                     │
│  GitHub: https://github.com/nf-core/sarek           │
│  Germline: HaplotypeCaller, DeepVariant, Strelka,   │
│            Manta, TIDDIT                            │
│  Somatic:  Mutect2, MuSE, Strelka, FreeBayes,      │
│            ASCAT, Control-FREEC                     │
│  Input:    WGS / WES / Targeted panel               │
│                                                     │
├─────────────────────────────────────────────────────┤
│            POST-FLIGHT (Databricks-native)           │
│  ┌───────────┐ ┌──────────┐ ┌────────────────────┐  │
│  │ Parse     │ │ Somatic/ │ │ Write calls to     │  │
│  │ Execution │ │ Germline │ │ Delta + trigger    │  │
│  │ Trace     │ │ QC Gates │ │ VEP/ANNOVAR v2     │  │
│  └───────────┘ └──────────┘ └────────────────────┘  │
└─────────────────────────────────────────────────────┘
```

**Key design principles:**
- **Zero fork**: nf-core/sarek runs unmodified via `-c eisai_sarek.config`
- **Institutional overlay**: Process selectors tune CPU/memory per Eisai compute
- **Audit trail**: Every run logged to shared Delta `nextflow_run_audit` table
- **Downstream integration**: Output VCFs feed directly into variant annotation v2 (VEP 115 + ANNOVAR)
- **Multi-caller support**: Run multiple callers simultaneously, compute cross-caller concordance

In [0]:
# Widget parameters — set by Databricks Jobs API or Streamlit form
dbutils.widgets.text("fastq_dir", "", "FASTQ Directory")
dbutils.widgets.text("bam_dir", "", "BAM Directory (alternative to FASTQ)")
dbutils.widgets.text("output_dir", "", "Output Directory")
dbutils.widgets.dropdown("genome", "GRCh38",
    ["GRCh38", "GRCh37", "GRCm38", "GRCm39"], "Reference Genome")
dbutils.widgets.text("tools", "mutect2,muse,strelka,vep", "Variant Callers (--tools)")
dbutils.widgets.dropdown("step", "mapping",
    ["mapping", "markduplicates", "recalibrate", "variant_calling", "annotate"], "Pipeline Step")
dbutils.widgets.dropdown("analysis_type", "somatic",
    ["somatic", "germline", "both"], "Analysis Type")
dbutils.widgets.text("pipeline_version", "3.5.1", "nf-core/sarek Version")
dbutils.widgets.text("extra_args", "", "Additional Nextflow Arguments")
dbutils.widgets.dropdown("qc_gate_enabled", "true", ["true", "false"], "Enable QC Gates")
dbutils.widgets.text("min_somatic_variants", "10", "Min Somatic Variants (QC)")
dbutils.widgets.text("max_titv_ratio", "3.5", "Max Ti/Tv Ratio (QC)")
dbutils.widgets.text("min_tumor_depth", "20", "Min Tumor Depth (QC)")
dbutils.widgets.dropdown("trigger_annotation", "true", ["true", "false"], "Trigger Annotation v2")
dbutils.widgets.text("catalog", "dhbl_discovery_us_dev", "Catalog")
dbutils.widgets.text("schema", "genesis_schema", "Schema")

# Retrieve parameter values
fastq_dir = dbutils.widgets.get("fastq_dir").strip()
bam_dir = dbutils.widgets.get("bam_dir").strip()
output_dir = dbutils.widgets.get("output_dir").strip()
genome = dbutils.widgets.get("genome")
tools = dbutils.widgets.get("tools").strip()
step = dbutils.widgets.get("step")
analysis_type = dbutils.widgets.get("analysis_type")
pipeline_version = dbutils.widgets.get("pipeline_version").strip()
extra_args = dbutils.widgets.get("extra_args").strip()
qc_gate_enabled = dbutils.widgets.get("qc_gate_enabled") == "true"
min_somatic_variants = int(dbutils.widgets.get("min_somatic_variants"))
max_titv_ratio = float(dbutils.widgets.get("max_titv_ratio"))
min_tumor_depth = int(dbutils.widgets.get("min_tumor_depth"))
trigger_annotation = dbutils.widgets.get("trigger_annotation") == "true"
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()

print(f"\u2705 Parameters loaded")
print(f"   Pipeline: nf-core/sarek v{pipeline_version}")
print(f"   Analysis: {analysis_type} | Step: {step}")
print(f"   Tools: {tools}")
print(f"   Genome: {genome}")

---
## \U0001f535 Layer 1 — Pre-Flight (Databricks-Native)

Validates inputs, auto-detects tumor/normal pairing, builds the nf-core/sarek samplesheet (with status column for somatic/germline), generates the institutional config overlay, and registers the run. **Zero pipeline modifications.**

In [0]:
import os, sys, glob, subprocess, re, json, csv
from datetime import datetime, timezone

# ── Validate Nextflow ──
try:
    nxf_out = subprocess.check_output(["nextflow", "-version"], stderr=subprocess.STDOUT, text=True)
    nxf_version = [l for l in nxf_out.strip().split("\n") if "version" in l.lower()]
    print(f"\u2705 Nextflow: {nxf_version[0].strip() if nxf_version else nxf_out.strip().split(chr(10))[0]}")
except FileNotFoundError:
    raise RuntimeError(
        "\u274c Nextflow not found. Install via: curl -s https://get.nextflow.io | bash\n"
        "   Then move to PATH: mv nextflow /usr/local/bin/"
    )

# ── Validate input paths ──
if not fastq_dir and not bam_dir:
    raise ValueError("\u274c Either fastq_dir or bam_dir must be provided")

input_mode = "fastq" if fastq_dir else "bam"

if input_mode == "fastq":
    if not os.path.isdir(fastq_dir):
        raise FileNotFoundError(f"\u274c FASTQ directory not found: {fastq_dir}")
    fq_files = glob.glob(os.path.join(fastq_dir, "**/*.fastq.gz"), recursive=True)
    fq_files += glob.glob(os.path.join(fastq_dir, "**/*.fq.gz"), recursive=True)
    if not fq_files:
        raise FileNotFoundError(f"\u274c No FASTQ files found in: {fastq_dir}")
    print(f"\u2705 Input mode: FASTQ ({len(fq_files)} files in {fastq_dir})")
else:
    if not os.path.isdir(bam_dir):
        raise FileNotFoundError(f"\u274c BAM directory not found: {bam_dir}")
    bam_files = glob.glob(os.path.join(bam_dir, "**/*.bam"), recursive=True)
    if not bam_files:
        raise FileNotFoundError(f"\u274c No BAM files found in: {bam_dir}")
    print(f"\u2705 Input mode: BAM ({len(bam_files)} files in {bam_dir})")

# ── Validate/create output directory ──
os.makedirs(output_dir, exist_ok=True)
print(f"\u2705 Output directory: {output_dir}")

# ── Summary ──
print(f"\n\u2139\ufe0f  Analysis type: {analysis_type}")
print(f"\u2139\ufe0f  Genome: {genome}")
print(f"\u2139\ufe0f  Tools: {tools}")
print(f"\u2139\ufe0f  Step: {step}")

In [0]:
def build_sarek_samplesheet(fastq_dir: str, bam_dir: str, output_path: str,
                            analysis_type: str) -> dict:
    """
    Build an nf-core/sarek samplesheet CSV.

    FASTQ input format: patient,sample,lane,fastq_1,fastq_2,status
    BAM input format:   patient,sample,bam,bai,status

    status: 0 = normal, 1 = tumor
    Tumor/normal pairing detected from naming convention:
      *_T_* or *_tumor* = tumor (status=1)
      *_N_* or *_normal* = normal (status=0)
    """
    rows = []

    def classify_status(filename: str, analysis_type: str) -> int:
        """Determine tumor (1) vs normal (0) status from filename."""
        if analysis_type == "germline":
            return 0
        fname = os.path.basename(filename).upper()
        tumor_patterns = ["_T_", "_TUMOR", "-T-", "-TUMOR", ".TUMOR", "_T."]
        if any(p in fname for p in tumor_patterns):
            return 1
        normal_patterns = ["_N_", "_NORMAL", "-N-", "-NORMAL", ".NORMAL", "_N."]
        if any(p in fname for p in normal_patterns):
            return 0
        # Default: tumor for somatic, normal for germline/both
        return 1 if analysis_type == "somatic" else 0

    def derive_patient(filename: str) -> str:
        """Extract patient ID from filename (common prefix before T/N indicator).

        Strategy: strip extensions, read indicator, lane suffix, then remove
        standalone T/N/tumor/normal segments (those bounded by delimiters).
        Requires a delimiter BEFORE the indicator to avoid matching inside
        words like 'PATIENT'.
        """
        base = os.path.basename(filename)
        # Remove extension and read indicator
        base = re.sub(r'(_R[12])?(\.[^.]+)*$', '', base)
        # Remove lane suffix first (so _T isn't confused with _L patterns)
        base = re.sub(r'[_\-\.]L\d+', '', base)
        # Remove standalone T/N/tumor/normal indicators
        # Key: require a delimiter [_\-\.] BEFORE the indicator (prevents
        # matching inside words like PATIENT, SENTINEL, CONTROL_N2, etc.)
        patient = re.sub(r'[_\-\.](T|N|tumor|normal)(?=[_\-\.]|$)', '', base, flags=re.IGNORECASE)
        # Clean up trailing/multiple underscores
        patient = re.sub(r'[_\-\.]+$', '', patient)
        patient = re.sub(r'[_\-\.]{2,}', '_', patient)
        return patient if patient else "patient_001"

    def derive_sample(filename: str) -> str:
        """Extract sample name from filename."""
        base = os.path.basename(filename)
        sample = re.sub(r'(_R[12])?(\.[^.]+)*$', '', base)
        sample = re.sub(r'[_\-\.]L\d+', '', sample)  # Remove lane suffix
        return sample if sample else "sample_001"

    def extract_lane(filename: str) -> str:
        """Extract lane from filename or default to L001."""
        match = re.search(r'[_\-\.]L(\d+)', os.path.basename(filename))
        return f"L{match.group(1).zfill(3)}" if match else "L001"

    if fastq_dir and os.path.isdir(fastq_dir):
        # FASTQ mode: patient,sample,lane,fastq_1,fastq_2,status
        fq_files = sorted(
            glob.glob(os.path.join(fastq_dir, "**/*.fastq.gz"), recursive=True) +
            glob.glob(os.path.join(fastq_dir, "**/*.fq.gz"), recursive=True)
        )

        # Pair R1/R2
        r1_files = [f for f in fq_files if re.search(r'_R1[_.]', os.path.basename(f))]
        for r1 in r1_files:
            r2 = re.sub(r'_R1([_.])', r'_R2\1', r1)
            if not os.path.exists(r2):
                r2 = ""  # Single-end
            patient = derive_patient(r1)
            sample = derive_sample(r1)
            lane = extract_lane(r1)
            status = classify_status(r1, analysis_type)
            rows.append({
                "patient": patient,
                "sample": sample,
                "lane": lane,
                "fastq_1": r1,
                "fastq_2": r2,
                "status": status
            })

        # Handle unpaired files (no _R1/_R2 pattern)
        paired_files = set()
        for r1 in r1_files:
            paired_files.add(r1)
            r2 = re.sub(r'_R1([_.])', r'_R2\1', r1)
            if os.path.exists(r2):
                paired_files.add(r2)
        unpaired = [f for f in fq_files if f not in paired_files
                    and not re.search(r'_R2[_.]', os.path.basename(f))]
        for f in unpaired:
            patient = derive_patient(f)
            sample = derive_sample(f)
            lane = extract_lane(f)
            status = classify_status(f, analysis_type)
            rows.append({
                "patient": patient,
                "sample": sample,
                "lane": lane,
                "fastq_1": f,
                "fastq_2": "",
                "status": status
            })

        header = ["patient", "sample", "lane", "fastq_1", "fastq_2", "status"]

    elif bam_dir and os.path.isdir(bam_dir):
        # BAM mode: patient,sample,bam,bai,status
        bam_files = sorted(glob.glob(os.path.join(bam_dir, "**/*.bam"), recursive=True))
        for bam in bam_files:
            bai = bam + ".bai"
            if not os.path.exists(bai):
                bai = bam.replace(".bam", ".bai")
            patient = derive_patient(bam)
            sample = derive_sample(bam)
            status = classify_status(bam, analysis_type)
            rows.append({
                "patient": patient,
                "sample": sample,
                "bam": bam,
                "bai": bai if os.path.exists(bai) else "",
                "status": status
            })

        header = ["patient", "sample", "bam", "bai", "status"]

    # Write CSV
    with open(output_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=header)
        writer.writeheader()
        writer.writerows(rows)

    # Compute stats
    n_tumor = sum(1 for r in rows if r["status"] == 1)
    n_normal = sum(1 for r in rows if r["status"] == 0)
    patients = set(r["patient"] for r in rows)

    stats = {
        "n_patients": len(patients),
        "n_samples": len(rows),
        "n_tumor": n_tumor,
        "n_normal": n_normal,
        "input_mode": "fastq" if fastq_dir else "bam",
    }
    return stats


# Build the samplesheet
samplesheet_path = os.path.join(output_dir, "sarek_samplesheet.csv")
sample_stats = build_sarek_samplesheet(fastq_dir, bam_dir, samplesheet_path, analysis_type)

print(f"\u2705 Samplesheet written: {samplesheet_path}")
print(f"   Patients: {sample_stats['n_patients']}")
print(f"   Samples:  {sample_stats['n_samples']} "
      f"(tumor={sample_stats['n_tumor']}, normal={sample_stats['n_normal']})")
print(f"   Mode:     {sample_stats['input_mode'].upper()}")

if analysis_type in ("somatic", "both") and sample_stats["n_normal"] == 0:
    print("\u26a0\ufe0f  WARNING: Somatic analysis requested but no normal samples detected.")
    print("   MuSE and Mutect2 require matched tumor/normal pairs.")
    print("   Check naming convention: *_T_* for tumor, *_N_* for normal.")

In [0]:
def generate_eisai_sarek_config(output_dir: str, tools: str) -> str:
    """
    Generate eisai_sarek.config — institutional overlay for nf-core/sarek.

    Provides per-process resource tuning for Eisai compute infrastructure.
    All processes get retry logic to handle transient failures.
    """
    config_content = """\
// ============================================================================
// Genesis Workbench — Eisai institutional config for nf-core/sarek
// Generated: {timestamp}
// Tools: {tools}
// ============================================================================

profiles {{
    conda {{
        conda.enabled = true
        docker.enabled = false
        singularity.enabled = false
    }}
}}

process {{
    // ---- Global defaults ----
    errorStrategy = 'retry'
    maxRetries = 2

    // ---- Alignment ----
    withName: 'BWAMEM2_MEM' {{
        cpus = 16
        memory = '32.GB'
        time = '24.h'
    }}
    withName: 'BWA_MEM' {{
        cpus = 16
        memory = '32.GB'
        time = '24.h'
    }}

    // ---- Preprocessing ----
    withName: 'MARKDUPLICATES' {{
        cpus = 4
        memory = '16.GB'
        time = '8.h'
    }}
    withName: 'BASERECALIBRATOR' {{
        cpus = 4
        memory = '16.GB'
        time = '8.h'
    }}
    withName: 'APPLYBQSR' {{
        cpus = 4
        memory = '16.GB'
        time = '8.h'
    }}

    // ---- Germline Callers ----
    withName: 'HAPLOTYPECALLER' {{
        cpus = 4
        memory = '16.GB'
        time = '12.h'
    }}
    withName: 'DEEPVARIANT' {{
        cpus = 8
        memory = '32.GB'
        time = '12.h'
    }}
    withName: 'STRELKA_GERMLINE' {{
        cpus = 8
        memory = '16.GB'
        time = '8.h'
    }}
    withName: 'MANTA_GERMLINE' {{
        cpus = 8
        memory = '16.GB'
        time = '8.h'
    }}
    withName: 'TIDDIT_SV' {{
        cpus = 4
        memory = '16.GB'
        time = '8.h'
    }}

    // ---- Somatic Callers ----
    withName: 'MUTECT2' {{
        cpus = 4
        memory = '16.GB'
        time = '12.h'
    }}
    withName: 'MUSE_CALL' {{
        cpus = 4
        memory = '8.GB'
        time = '12.h'
    }}
    withName: 'MUSE_SUMP' {{
        cpus = 4
        memory = '8.GB'
        time = '4.h'
    }}
    withName: 'STRELKA_SOMATIC' {{
        cpus = 8
        memory = '16.GB'
        time = '8.h'
    }}
    withName: 'FREEBAYES' {{
        cpus = 4
        memory = '16.GB'
        time = '12.h'
    }}
    withName: 'MANTA_SOMATIC' {{
        cpus = 8
        memory = '16.GB'
        time = '8.h'
    }}

    // ---- CNV / SV Callers ----
    withName: 'ASCAT' {{
        cpus = 2
        memory = '8.GB'
        time = '8.h'
    }}
    withName: 'CONTROLFREEC' {{
        cpus = 8
        memory = '16.GB'
        time = '12.h'
    }}

    // ---- Annotation ----
    withName: 'VEP' {{
        cpus = 4
        memory = '8.GB'
        time = '4.h'
    }}
    withName: 'SNPEFF' {{
        cpus = 4
        memory = '8.GB'
        time = '4.h'
    }}

    // ---- QC & Utilities ----
    withName: 'FASTQC' {{
        cpus = 4
        memory = '8.GB'
        time = '4.h'
    }}
    withName: 'FASTP' {{
        cpus = 4
        memory = '8.GB'
        time = '4.h'
    }}
    withName: 'MULTIQC' {{
        cpus = 2
        memory = '8.GB'
        time = '2.h'
    }}
    withName: 'SAMTOOLS_.*' {{
        cpus = 4
        memory = '8.GB'
        time = '4.h'
    }}
}}
""".format(timestamp=datetime.now(timezone.utc).isoformat(), tools=tools)

    config_path = os.path.join(output_dir, "eisai_sarek.config")
    with open(config_path, "w") as f:
        f.write(config_content)

    return config_path


# Generate config
eisai_config_path = generate_eisai_sarek_config(output_dir, tools)
print(f"\u2705 Config written: {eisai_config_path}")
print(f"   Process selectors: alignment, preprocessing, germline callers,")
print(f"   somatic callers, CNV/SV, annotation, QC utilities")
print(f"   Error strategy: retry (max 2 attempts)")

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    TimestampType, DoubleType, BooleanType,
)

# Reuse the same audit table as scrnaseq/rnaseq — pipeline column distinguishes them
AUDIT_TABLE = f"{catalog}.{schema}.nextflow_run_audit"

run_started_at = datetime.now(timezone.utc)
run_id = run_started_at.strftime("%Y%m%d_%H%M%S") + "_sarek"

audit_row = Row(
    run_id=run_id,
    pipeline="sarek",
    pipeline_version=pipeline_version,
    status="started",
    started_at=run_started_at,
    completed_at=None,
    elapsed_minutes=None,
    input_mode=input_mode,
    input_path=fastq_dir if fastq_dir else bam_dir,
    output_path=output_dir,
    genome=genome,
    tools=tools,
    step=step,
    analysis_type=analysis_type,
    n_patients=sample_stats["n_patients"],
    n_samples=sample_stats["n_samples"],
    n_tumor=sample_stats["n_tumor"],
    n_normal=sample_stats["n_normal"],
    qc_passed=None,
    qc_reasons=None,
    error_message=None,
    triggered_by=dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get(),
)

# Create table if not exists (idempotent)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {AUDIT_TABLE} (
        run_id STRING,
        pipeline STRING,
        pipeline_version STRING,
        status STRING,
        started_at TIMESTAMP,
        completed_at TIMESTAMP,
        elapsed_minutes DOUBLE,
        input_mode STRING,
        input_path STRING,
        output_path STRING,
        genome STRING,
        tools STRING,
        step STRING,
        analysis_type STRING,
        n_patients INT,
        n_samples INT,
        n_tumor INT,
        n_normal INT,
        qc_passed BOOLEAN,
        qc_reasons STRING,
        error_message STRING,
        triggered_by STRING
    )
""")

# Explicit schema required — None values prevent type inference
audit_schema = StructType([
    StructField('run_id', StringType()),
    StructField('pipeline', StringType()),
    StructField('pipeline_version', StringType()),
    StructField('status', StringType()),
    StructField('started_at', TimestampType()),
    StructField('completed_at', TimestampType()),
    StructField('elapsed_minutes', DoubleType()),
    StructField('input_mode', StringType()),
    StructField('input_path', StringType()),
    StructField('output_path', StringType()),
    StructField('genome', StringType()),
    StructField('tools', StringType()),
    StructField('step', StringType()),
    StructField('analysis_type', StringType()),
    StructField('n_patients', IntegerType()),
    StructField('n_samples', IntegerType()),
    StructField('n_tumor', IntegerType()),
    StructField('n_normal', IntegerType()),
    StructField('qc_passed', BooleanType()),
    StructField('qc_reasons', StringType()),
    StructField('error_message', StringType()),
    StructField('triggered_by', StringType()),
])

# Insert starting record
audit_df = spark.createDataFrame([audit_row], schema=audit_schema)
audit_df.write.mode("append").saveAsTable(AUDIT_TABLE)

print(f"\u2705 Run registered: {run_id}")
print(f"   Audit table: {AUDIT_TABLE}")
print(f"   Status: started")

---
## \U0001f7e2 Layer 2 — nf-core/sarek (Black Box)

Runs as an opaque subprocess. The only interface points are `-c eisai_sarek.config` and standard nf-core params (`--tools`, `--step`, `--genome`). **ZERO modifications** to the pipeline.

In [0]:
import time as _time

# Build the nf-core/sarek command
cmd = [
    "nextflow", "run",
    "nf-core/sarek",
    "-c", eisai_config_path,       # institutional overlay
    "-r", pipeline_version,
    "-profile", "conda",
    "--input", samplesheet_path,
    "--outdir", output_dir,
    "--genome", genome,
    "--tools", tools,
    "--step", step,
]

# Append extra arguments if provided
if extra_args:
    cmd.extend(extra_args.split())

print(f"\U0001f680 Launching nf-core/sarek v{pipeline_version}")
print(f"   Command: {' '.join(cmd)}")
print(f"   Started: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
print("\u2500" * 60)

# Stream stdout/stderr in real-time
nf_start = _time.time()
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

stdout_lines = []
for line in proc.stdout:
    line = line.rstrip()
    stdout_lines.append(line)
    # Print progress lines (executor, process completion)
    if any(kw in line for kw in ["executor", "process", "Completed", "ERROR", "WARN", "finished"]):
        print(line)

proc.wait()
nf_exit_code = proc.returncode
nf_elapsed = (_time.time() - nf_start) / 60  # minutes

print("\u2500" * 60)
if nf_exit_code == 0:
    print(f"\u2705 nf-core/sarek completed successfully in {nf_elapsed:.1f} min")
else:
    print(f"\u274c nf-core/sarek FAILED (exit code {nf_exit_code}) after {nf_elapsed:.1f} min")
    # Print last 20 lines for debugging
    print("\n\u2500\u2500 Last 20 lines of output \u2500\u2500")
    for line in stdout_lines[-20:]:
        print(f"  {line}")

---
## \U0001f7e1 Layer 3 — Post-Flight (Databricks-Native)

Parses execution traces, locates variant call files per caller, runs somatic/germline QC gates (variant count, Ti/Tv ratio, tumor depth, cross-caller concordance), writes calls to Delta, and optionally triggers the variant annotation v2 pipeline (VEP 115 + ANNOVAR). **Zero nf-core modifications.**

In [0]:
import pandas as pd

TRACE_TABLE = f"{catalog}.{schema}.nextflow_execution_traces"
trace_path = os.path.join(output_dir, "pipeline_info", "execution_trace*.txt")

trace_df = None
trace_files = glob.glob(trace_path)
if trace_files:
    trace_file = sorted(trace_files)[-1]  # Latest trace
    try:
        trace_pdf = pd.read_csv(trace_file, sep="\t")
        trace_pdf["run_id"] = run_id
        trace_pdf["pipeline"] = "sarek"
        trace_pdf["pipeline_version"] = pipeline_version

        trace_df = spark.createDataFrame(trace_pdf)

        # Create table if not exists
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {TRACE_TABLE} (
                task_id STRING,
                hash STRING,
                native_id STRING,
                name STRING,
                status STRING,
                exit STRING,
                submit STRING,
                duration STRING,
                realtime STRING,
                `%cpu` STRING,
                peak_rss STRING,
                peak_vmem STRING,
                rchar STRING,
                wchar STRING,
                run_id STRING,
                pipeline STRING,
                pipeline_version STRING
            )
        """)

        trace_df.write.mode("append").saveAsTable(TRACE_TABLE)
        n_tasks = trace_pdf.shape[0]
        n_failed = trace_pdf[trace_pdf["status"] != "COMPLETED"].shape[0]
        print(f"\u2705 Execution trace: {n_tasks} tasks parsed ({n_failed} non-COMPLETED)")
        print(f"   Saved to: {TRACE_TABLE}")
    except Exception as e:
        print(f"\u26a0\ufe0f  Could not parse trace file: {e}")
else:
    print(f"\u26a0\ufe0f  No execution trace found at: {trace_path}")
    print("   (Expected for failed runs or --step variant_calling with cached work)")

In [0]:
def find_sarek_outputs(output_dir: str, tools: str) -> dict:
    """
    Locate key output files from a completed nf-core/sarek run.

    Sarek output structure:
    - {outdir}/variant_calling/{tool}/{sample}/*.vcf.gz
    - {outdir}/preprocessing/recalibrated/{sample}/*.bam
    - {outdir}/reports/multiqc/multiqc_report.html
    """
    outputs = {
        "vcfs": {},        # tool -> list of VCF paths
        "bams": [],        # recalibrated BAMs
        "multiqc": None,   # MultiQC report path
        "summary": {},     # tool -> variant count
    }

    # Variant callers to check
    tool_list = [t.strip().lower() for t in tools.split(",")]
    # Map common tool names to output directory names
    tool_dirs = {
        "mutect2": "mutect2",
        "muse": "muse",
        "strelka": "strelka",
        "freebayes": "freebayes",
        "haplotypecaller": "haplotypecaller",
        "deepvariant": "deepvariant",
        "manta": "manta",
        "tiddit": "tiddit",
        "ascat": "ascat",
        "controlfreec": "controlfreec",
    }

    vc_base = os.path.join(output_dir, "variant_calling")
    for tool_name, dir_name in tool_dirs.items():
        if tool_name in tool_list or tool_name in ["mutect2", "strelka", "muse"]:
            tool_path = os.path.join(vc_base, dir_name)
            if os.path.isdir(tool_path):
                vcfs = glob.glob(os.path.join(tool_path, "**/*.vcf.gz"), recursive=True)
                # Filter to only PASS/final VCFs (exclude intermediate)
                final_vcfs = [v for v in vcfs if "filtered" in v.lower()
                              or not any(x in v.lower() for x in ["unfiltered", "stats"])]
                if not final_vcfs:
                    final_vcfs = vcfs  # Fall back to all VCFs
                outputs["vcfs"][tool_name] = final_vcfs
                outputs["summary"][tool_name] = len(final_vcfs)

    # Recalibrated BAMs
    recal_path = os.path.join(output_dir, "preprocessing", "recalibrated")
    if os.path.isdir(recal_path):
        outputs["bams"] = glob.glob(os.path.join(recal_path, "**/*.bam"), recursive=True)

    # MultiQC report
    mqc_path = os.path.join(output_dir, "multiqc", "multiqc_report.html")
    if not os.path.exists(mqc_path):
        mqc_path = os.path.join(output_dir, "reports", "multiqc", "multiqc_report.html")
    if os.path.exists(mqc_path):
        outputs["multiqc"] = mqc_path

    return outputs


# Locate outputs
if nf_exit_code == 0:
    sarek_outputs = find_sarek_outputs(output_dir, tools)

    print(f"\u2705 Variant call outputs located:")
    for tool_name, vcf_list in sarek_outputs["vcfs"].items():
        print(f"   {tool_name}: {len(vcf_list)} VCF file(s)")
    print(f"   Recalibrated BAMs: {len(sarek_outputs['bams'])}")
    print(f"   MultiQC: {'\u2705' if sarek_outputs['multiqc'] else '\u274c not found'}")
else:
    sarek_outputs = {"vcfs": {}, "bams": [], "multiqc": None, "summary": {}}
    print(f"\u26a0\ufe0f  Skipping output location (pipeline failed)")

In [0]:
import gzip

def count_pass_variants(vcf_path: str) -> dict:
    """
    Count PASS variants in a VCF and compute Ti/Tv ratio.

    Returns: {total, pass_count, snvs, indels, ti, tv, titv_ratio, depths}
    """
    stats = {"total": 0, "pass_count": 0, "snvs": 0, "indels": 0,
             "ti": 0, "tv": 0, "titv_ratio": 0.0, "depths": []}

    transitions = {("A", "G"), ("G", "A"), ("C", "T"), ("T", "C")}
    opener = gzip.open if vcf_path.endswith(".gz") else open

    try:
        with opener(vcf_path, "rt") as f:
            for line in f:
                if line.startswith("#"):
                    continue
                stats["total"] += 1
                fields = line.strip().split("\t")
                if len(fields) < 8:
                    continue

                filt = fields[6]
                if filt not in ("PASS", "."):
                    continue
                stats["pass_count"] += 1

                ref, alt = fields[3], fields[4].split(",")[0]

                if len(ref) == 1 and len(alt) == 1:
                    stats["snvs"] += 1
                    if (ref.upper(), alt.upper()) in transitions:
                        stats["ti"] += 1
                    else:
                        stats["tv"] += 1
                else:
                    stats["indels"] += 1

                # Extract depth from INFO or FORMAT
                info = fields[7]
                dp_match = re.search(r'DP=(\d+)', info)
                if dp_match:
                    stats["depths"].append(int(dp_match.group(1)))
                elif len(fields) > 9:
                    # Try FORMAT field
                    fmt_keys = fields[8].split(":")
                    if "DP" in fmt_keys:
                        dp_idx = fmt_keys.index("DP")
                        # Use first tumor sample (last column for Mutect2)
                        sample_vals = fields[-1].split(":")
                        if dp_idx < len(sample_vals) and sample_vals[dp_idx].isdigit():
                            stats["depths"].append(int(sample_vals[dp_idx]))

    except Exception as e:
        print(f"  \u26a0\ufe0f  Error parsing {vcf_path}: {e}")

    if stats["tv"] > 0:
        stats["titv_ratio"] = stats["ti"] / stats["tv"]

    return stats


# Run QC gates
qc_passed = True
qc_reasons = []

if nf_exit_code != 0:
    qc_passed = False
    qc_reasons.append(f"Pipeline failed with exit code {nf_exit_code}")

elif qc_gate_enabled:
    caller_stats = {}

    for tool_name, vcf_list in sarek_outputs["vcfs"].items():
        tool_stats = {"total": 0, "pass_count": 0, "snvs": 0, "indels": 0,
                      "ti": 0, "tv": 0, "titv_ratio": 0.0, "depths": []}
        for vcf in vcf_list:
            s = count_pass_variants(vcf)
            tool_stats["total"] += s["total"]
            tool_stats["pass_count"] += s["pass_count"]
            tool_stats["snvs"] += s["snvs"]
            tool_stats["indels"] += s["indels"]
            tool_stats["ti"] += s["ti"]
            tool_stats["tv"] += s["tv"]
            tool_stats["depths"].extend(s["depths"])

        if tool_stats["tv"] > 0:
            tool_stats["titv_ratio"] = tool_stats["ti"] / tool_stats["tv"]
        caller_stats[tool_name] = tool_stats

    # ---- Somatic QC checks ----
    if analysis_type in ("somatic", "both"):
        for tool_name, stats in caller_stats.items():
            if tool_name in ("ascat", "controlfreec", "manta", "tiddit"):
                continue  # CNV/SV callers have different thresholds

            # Check minimum somatic variants
            if stats["pass_count"] < min_somatic_variants:
                qc_passed = False
                qc_reasons.append(
                    f"{tool_name}: only {stats['pass_count']} PASS variants "
                    f"(min={min_somatic_variants})")

            # Check Ti/Tv ratio (expected ~2.0-2.1 for WGS, flag if extreme)
            if stats["titv_ratio"] > max_titv_ratio and stats["snvs"] > 100:
                qc_passed = False
                qc_reasons.append(
                    f"{tool_name}: Ti/Tv={stats['titv_ratio']:.2f} "
                    f"exceeds max={max_titv_ratio}")

            # Check tumor depth
            if stats["depths"]:
                import numpy as np
                median_depth = np.median(stats["depths"])
                if median_depth < min_tumor_depth:
                    qc_passed = False
                    qc_reasons.append(
                        f"{tool_name}: median depth={median_depth:.0f}x "
                        f"(min={min_tumor_depth}x)")

            # Flag zero-variant callers
            if stats["total"] == 0:
                qc_passed = False
                qc_reasons.append(f"{tool_name}: produced 0 variants")

        # Cross-caller concordance (if multiple callers)
        snv_callers = [t for t, s in caller_stats.items()
                       if s["snvs"] > 0 and t not in ("ascat", "controlfreec", "manta", "tiddit")]
        if len(snv_callers) >= 2:
            print(f"\n\u2139\ufe0f  Cross-caller concordance ({len(snv_callers)} callers):")
            for t in snv_callers:
                print(f"   {t}: {caller_stats[t]['pass_count']} PASS variants "
                      f"(SNVs={caller_stats[t]['snvs']}, Indels={caller_stats[t]['indels']})")

    # ---- Germline QC checks ----
    if analysis_type in ("germline", "both"):
        for tool_name, stats in caller_stats.items():
            if tool_name in ("haplotypecaller", "deepvariant", "strelka"):
                if stats["pass_count"] < 1000:
                    qc_passed = False
                    qc_reasons.append(
                        f"{tool_name}: only {stats['pass_count']} germline variants "
                        f"(expected >1000 for WGS)")
                # Heterozygosity ratio check (het/hom should be ~1.5-2.0)
                # Simplified: check Ti/Tv ~2.0-2.1 for germline
                if stats["titv_ratio"] < 1.0 and stats["snvs"] > 100:
                    qc_passed = False
                    qc_reasons.append(
                        f"{tool_name}: Ti/Tv={stats['titv_ratio']:.2f} "
                        f"suspiciously low (expected ~2.0 for germline)")

    print(f"\n{'\u2705' if qc_passed else '\u274c'} QC Result: {'PASSED' if qc_passed else 'FAILED'}")
    if qc_reasons:
        for reason in qc_reasons:
            print(f"   \u274c {reason}")
    else:
        print("   All QC gates passed.")
        for tool_name, stats in caller_stats.items():
            print(f"   {tool_name}: {stats['pass_count']} PASS "
                  f"(Ti/Tv={stats['titv_ratio']:.2f}, "
                  f"median_depth={int(np.median(stats['depths'])) if stats['depths'] else 'N/A'}x)")
else:
    print("\u2139\ufe0f  QC gates disabled")
    caller_stats = {}

In [0]:
def parse_vcf_to_rows(vcf_path: str, caller: str, patient: str, sample: str) -> list:
    """
    Parse a VCF file into a list of row dicts for Delta ingestion.
    Extracts core variant fields and key INFO annotations.
    """
    rows = []
    opener = gzip.open if vcf_path.endswith(".gz") else open

    try:
        with opener(vcf_path, "rt") as f:
            for line in f:
                if line.startswith("#"):
                    continue
                fields = line.strip().split("\t")
                if len(fields) < 8:
                    continue

                chrom = fields[0]
                pos = int(fields[1])
                ref = fields[3]
                alt = fields[4]
                qual = fields[5]
                filt = fields[6]
                info = fields[7]

                # Extract key INFO fields
                dp_match = re.search(r'DP=(\d+)', info)
                af_match = re.search(r'AF=([\d.e-]+)', info)

                rows.append({
                    "chrom": chrom,
                    "pos": pos,
                    "ref": ref,
                    "alt": alt,
                    "qual": qual if qual != "." else None,
                    "filter": filt,
                    "caller": caller,
                    "patient": patient,
                    "sample": sample,
                    "depth": int(dp_match.group(1)) if dp_match else None,
                    "allele_freq": float(af_match.group(1)) if af_match else None,
                    "variant_key": f"{chrom}:{pos}:{ref}:{alt}",
                    "run_id": run_id,
                })
    except Exception as e:
        print(f"  \u26a0\ufe0f  Error parsing {vcf_path}: {e}")

    return rows


# Write variant calls to Delta
VARIANT_TABLE = f"{catalog}.{schema}.sarek_variant_calls"
SUMMARY_TABLE = f"{catalog}.{schema}.sarek_run_summary"

if nf_exit_code == 0 and sarek_outputs["vcfs"]:
    all_rows = []
    summary_rows = []

    for tool_name, vcf_list in sarek_outputs["vcfs"].items():
        tool_variant_count = 0
        for vcf_path in vcf_list:
            # Derive patient/sample from path
            # Expected: .../variant_calling/{tool}/{sample}/filename.vcf.gz
            path_parts = vcf_path.split(os.sep)
            try:
                tool_idx = path_parts.index(tool_name)
                sample_name = path_parts[tool_idx + 1] if tool_idx + 1 < len(path_parts) else "unknown"
            except ValueError:
                sample_name = "unknown"
            patient_name = sample_name.split("_")[0] if "_" in sample_name else sample_name

            rows = parse_vcf_to_rows(vcf_path, tool_name, patient_name, sample_name)
            all_rows.extend(rows)
            tool_variant_count += len(rows)

        summary_rows.append({
            "run_id": run_id,
            "caller": tool_name,
            "total_variants": tool_variant_count,
            "pass_variants": caller_stats.get(tool_name, {}).get("pass_count", 0),
            "snvs": caller_stats.get(tool_name, {}).get("snvs", 0),
            "indels": caller_stats.get(tool_name, {}).get("indels", 0),
            "titv_ratio": caller_stats.get(tool_name, {}).get("titv_ratio", 0.0),
            "analysis_type": analysis_type,
            "genome": genome,
            "pipeline_version": pipeline_version,
        })

    # Write variant calls
    if all_rows:
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {VARIANT_TABLE} (
                chrom STRING,
                pos BIGINT,
                ref STRING,
                alt STRING,
                qual STRING,
                filter STRING,
                caller STRING,
                patient STRING,
                sample STRING,
                depth INT,
                allele_freq DOUBLE,
                variant_key STRING,
                run_id STRING
            )
            PARTITIONED BY (caller, patient)
        """)

        variants_df = spark.createDataFrame(all_rows)
        variants_df.write.mode("append").saveAsTable(VARIANT_TABLE)
        print(f"\u2705 Wrote {len(all_rows)} variants to {VARIANT_TABLE}")
    else:
        print("\u26a0\ufe0f  No variants parsed from VCF files")

    # Write run summary
    if summary_rows:
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {SUMMARY_TABLE} (
                run_id STRING,
                caller STRING,
                total_variants INT,
                pass_variants INT,
                snvs INT,
                indels INT,
                titv_ratio DOUBLE,
                analysis_type STRING,
                genome STRING,
                pipeline_version STRING
            )
        """)

        summary_df = spark.createDataFrame(summary_rows)
        summary_df.write.mode("append").saveAsTable(SUMMARY_TABLE)
        print(f"\u2705 Run summary saved to {SUMMARY_TABLE}")
else:
    print("\u26a0\ufe0f  Skipping Delta write (pipeline failed or no VCFs found)")

In [0]:
run_completed_at = datetime.now(timezone.utc)
run_elapsed = (run_completed_at - run_started_at).total_seconds() / 60

final_status = (
    "succeeded" if nf_exit_code == 0 and qc_passed
    else "succeeded_qc_failed" if nf_exit_code == 0 and not qc_passed
    else "failed"
)

# Escape single quotes for safe SQL interpolation
def _sql_esc(val):
    """Escape single quotes for SQL string literals."""
    return str(val).replace("'", "''") if val else ""

qc_reasons_str = _sql_esc('; '.join(qc_reasons) if qc_reasons else '')
error_msg_str = _sql_esc(stdout_lines[-1] if nf_exit_code != 0 and stdout_lines else '')

# Update audit table
spark.sql(f"""
    UPDATE {AUDIT_TABLE}
    SET status = '{_sql_esc(final_status)}',
        completed_at = '{run_completed_at.isoformat()}',
        elapsed_minutes = {run_elapsed:.2f},
        qc_passed = {str(qc_passed).lower()},
        qc_reasons = '{qc_reasons_str}',
        error_message = '{error_msg_str}'
    WHERE run_id = '{_sql_esc(run_id)}'
""")

print(f"\u2705 Audit table updated: status={final_status}, elapsed={run_elapsed:.1f} min")

# ---- Trigger Variant Annotation v2 ----
if trigger_annotation and nf_exit_code == 0 and qc_passed:
    # Collect all PASS VCF paths for annotation
    annotation_vcfs = []
    for tool_name, vcf_list in sarek_outputs["vcfs"].items():
        annotation_vcfs.extend(vcf_list)

    if annotation_vcfs:
        # Use the first VCF (or a merged VCF if available)
        primary_vcf = annotation_vcfs[0]
        print(f"\n\U0001f517 Triggering Variant Annotation v2 pipeline...")
        print(f"   VCF: {primary_vcf}")
        print(f"   Annotation notebook: modules/disease_biology/variant_annotation/"
              f"variant_annotation_v2/notebooks/01_vep_annovar_annotation")

        try:
            # Trigger downstream annotation notebook
            annotation_result = dbutils.notebook.run(
                "../../variant_annotation/variant_annotation_v2/notebooks/01_vep_annovar_annotation",
                timeout_seconds=7200,  # 2 hour timeout
                arguments={
                    "catalog": catalog,
                    "schema": schema,
                    "vcf_path": primary_vcf,
                    "genome_build": genome,
                    "annotation_tools": "both",
                    "run_name": f"sarek_{run_id}",
                    "user_email": dbutils.notebook.entry_point.getDbutils()
                        .notebook().getContext().userName().get(),
                }
            )
            print(f"   \u2705 Annotation completed: {annotation_result}")
        except Exception as e:
            print(f"   \u26a0\ufe0f  Annotation trigger failed: {e}")
            print(f"   Run manually with: vcf_path='{primary_vcf}'")
    else:
        print("\u26a0\ufe0f  No VCFs available for annotation")

elif trigger_annotation and nf_exit_code == 0 and not qc_passed:
    print(f"\n\u26a0\ufe0f  Annotation NOT triggered (QC failed)")
    print(f"   Fix QC issues and re-run, or manually trigger with:")
    for tool_name, vcf_list in sarek_outputs["vcfs"].items():
        if vcf_list:
            print(f"   vcf_path='{vcf_list[0]}'")
            break
else:
    print(f"\n\u2139\ufe0f  Annotation trigger: {'disabled' if not trigger_annotation else 'skipped (pipeline failed)'}")

In [0]:
# =============================================================================
# Lineage Logging — Metadata Tier Integration
# Records assets + edges in the genesis_schema metadata tier tables so that
# downstream analyses (VEP/ANNOVAR, clinical reports) can trace provenance
# back to raw FASTQs and through each variant caller.
# =============================================================================

try:
    from genesis_workbench.lineage import LineageLogger

    user_email = (
        dbutils.notebook.entry_point.getDbutils()
        .notebook().getContext().userName().get()
    )

    lineage = LineageLogger(
        module="sarek",
        run_id=run_id,
        user_email=user_email,
        run_source="nextflow",
        catalog=catalog,
        schema=schema,
    )

    # ── Register input assets ──
    input_assets = []

    if fastq_dir and os.path.isdir(fastq_dir):
        # Bronze-tier: raw FASTQs
        fastq_asset = lineage.register_asset(
            path=fastq_dir,
            asset_type="volume_file",
            tier="bronze",
            format="fastq.gz",
            display_name=f"Sarek Input FASTQs ({project_name})",
            description=f"{n_patients} patients, {n_samples} samples ({n_tumor} tumor, {n_normal} normal)",
            tags={
                "genome": genome,
                "analysis_type": analysis_type,
                "n_patients": str(n_patients),
                "n_samples": str(n_samples),
                "n_tumor": str(n_tumor),
                "n_normal": str(n_normal),
            },
        )
        input_assets.append(fastq_asset)

    elif bam_dir and os.path.isdir(bam_dir):
        # Gold-tier: pre-aligned BAMs (already processed)
        bam_asset = lineage.register_asset(
            path=bam_dir,
            asset_type="volume_file",
            tier="gold",
            format="bam",
            display_name=f"Sarek Input BAMs ({project_name})",
            description=f"{n_patients} patients, {n_samples} samples",
            tags={"genome": genome, "analysis_type": analysis_type},
        )
        input_assets.append(bam_asset)

    # ── Register output assets: VCFs per caller (gold tier) ──
    vcf_assets = []
    if nf_exit_code == 0 and 'sarek_outputs' in dir() and sarek_outputs:
        for caller_name, vcf_list in sarek_outputs.get("vcfs", {}).items():
            for vcf_path in vcf_list[:3]:  # Cap at 3 per caller to avoid bloat
                vcf_asset = lineage.register_asset(
                    path=vcf_path,
                    asset_type="nextflow_output",
                    tier="gold",
                    format="vcf.gz",
                    display_name=f"{caller_name} VCF ({os.path.basename(vcf_path)})",
                    tags={
                        "caller": caller_name,
                        "genome": genome,
                        "pipeline": "nf-core/sarek",
                        "pipeline_version": pipeline_version,
                        "analysis_type": analysis_type,
                    },
                )
                vcf_assets.append((caller_name, vcf_asset))

    # ── Register output asset: variant Delta table (platinum) ──
    variant_table_asset = None
    VARIANT_TABLE = f"{catalog}.{schema}.sarek_variant_calls"
    if nf_exit_code == 0 and vcf_assets:
        total_variants = sum(
            qc_results.get(c, {}).get("pass_count", 0)
            for c in sarek_outputs.get("vcfs", {}).keys()
        ) if 'qc_results' in dir() and qc_results else None

        variant_table_asset = lineage.register_asset(
            path=VARIANT_TABLE,
            asset_type="delta_table",
            tier="platinum",
            display_name=f"Sarek Variant Calls ({project_name})",
            description=f"Parsed VCF rows from {len(sarek_outputs.get('vcfs', {}))} callers",
            row_count=total_variants,
            tags={
                "callers": tools,
                "genome": genome,
                "analysis_type": analysis_type,
                "pipeline_version": pipeline_version,
            },
        )

    # ── Record lineage edges ──
    # Inputs → VCFs (consumed_by)
    for input_asset in input_assets:
        for caller_name, vcf_asset in vcf_assets:
            lineage.link(
                source=input_asset,
                target=vcf_asset,
                relationship="consumed_by",
                step=caller_name,
            )

    # VCFs → variant Delta table (consumed_by)
    if variant_table_asset:
        for caller_name, vcf_asset in vcf_assets:
            lineage.link(
                source=vcf_asset,
                target=variant_table_asset,
                relationship="consumed_by",
                step="parse_vcf_to_delta",
            )

    # If annotation was triggered, record the trigger edge
    if trigger_annotation and nf_exit_code == 0 and qc_passed and vcf_assets:
        # Register the downstream annotation target as a placeholder
        annotation_asset = lineage.register_asset(
            path=f"{catalog}.{schema}.variant_annotations_v2",
            asset_type="delta_table",
            tier="platinum",
            display_name="Variant Annotations v2 (VEP + ANNOVAR)",
            tags={"genome": genome, "vep_version": "115"},
        )
        # First VCF triggered annotation
        _first_caller, first_vcf = vcf_assets[0]
        lineage.link(
            source=first_vcf,
            target=annotation_asset,
            relationship="triggered_by",
            step="auto_trigger_vep_annovar",
        )

    # ── Register the run ──
    lineage.register_run(
        run_name=f"sarek_{project_name}",
        experiment_name="nf-core/sarek",
        status=final_status,
        parameters={
            "fastq_dir": fastq_dir or "",
            "bam_dir": bam_dir or "",
            "output_dir": output_dir,
            "genome": genome,
            "tools": tools,
            "step": step,
            "analysis_type": analysis_type,
            "pipeline_version": pipeline_version,
        },
        metrics={
            "elapsed_minutes": run_elapsed,
            "n_patients": float(n_patients),
            "n_samples": float(n_samples),
            "n_tumor": float(n_tumor),
            "n_normal": float(n_normal),
            "n_callers": float(len(sarek_outputs.get("vcfs", {}))) if 'sarek_outputs' in dir() and sarek_outputs else 0.0,
        },
        start_time=run_started_at,
        end_time=run_completed_at,
        tags={
            "project_name": project_name,
            "qc_passed": str(qc_passed),
            "trigger_annotation": str(trigger_annotation),
            "analysis_type": analysis_type,
        },
    )

    # ── Flush to Delta ──
    result = lineage.flush()
    print(f"\n\U0001f4cb Lineage logged: {result['assets_written']} assets, "
          f"{result['edges_written']} edges, {result['run_written']} run")

except ImportError:
    print("\u26a0\ufe0f  genesis_workbench.lineage not available — skipping lineage logging")
    print("   Install wheel v0.1.3+ to enable metadata tier integration")
except Exception as e:
    # Non-fatal: lineage logging should never break the variant calling pipeline
    print(f"\u26a0\ufe0f  Lineage logging failed (non-fatal): {e}")

In [0]:
# Defensive defaults for variables that depend on successful pipeline completion
if 'caller_stats' not in dir():
    caller_stats = {}
if 'qc_passed' not in dir():
    qc_passed = False
if 'qc_reasons' not in dir():
    qc_reasons = ["Pipeline did not produce variant calls"]
if 'sarek_outputs' not in dir():
    sarek_outputs = {"vcfs": [], "multiqc": None}
if 'nf_exit_code' not in dir():
    nf_exit_code = -1
if 'final_status' not in dir():
    final_status = "unknown"
if 'run_elapsed' not in dir():
    from datetime import datetime, timezone
    run_elapsed = (datetime.now(timezone.utc) - run_started_at).total_seconds() / 60
if 'VARIANT_TABLE' not in dir():
    VARIANT_TABLE = f"{catalog}.{schema}.sarek_variant_calls"
if 'SUMMARY_TABLE' not in dir():
    SUMMARY_TABLE = f"{catalog}.{schema}.sarek_qc_summary"
if 'trace_df' not in dir():
    trace_df = None
if 'TRACE_TABLE' not in dir():
    TRACE_TABLE = f"{catalog}.{schema}.nextflow_execution_traces"

print("\u2554" + "\u2550"*58 + "\u2557")
print("\u2551  Sarek Variant Calling Sandwich Wrapper \u2014 Run Complete" + " "*2 + "\u2551")
print("\u2560" + "\u2550"*58 + "\u2563")
print(f"\u2551  Run ID:    {run_id:<45}\u2551")
print(f"\u2551  Pipeline:  nf-core/sarek v{pipeline_version:<35}\u2551")
print(f"\u2551  Analysis:  {analysis_type:<45}\u2551")
print(f"\u2551  Tools:     {tools:<45}\u2551")
print(f"\u2551  Genome:    {genome:<45}\u2551")
print(f"\u2551  Step:      {step:<45}\u2551")
print("\u2560" + "\u2550"*58 + "\u2563")
print(f"\u2551  Status:    {final_status:<45}\u2551")
print(f"\u2551  Elapsed:   {run_elapsed:.1f} min{' '*(40-len(f'{run_elapsed:.1f} min'))}\u2551")
print(f"\u2551  QC:        {'PASSED \u2705' if qc_passed else 'FAILED \u274c':<45}\u2551")
print("\u2560" + "\u2550"*58 + "\u2563")
print(f"\u2551  Variants per caller:{' '*37}\u2551")
if caller_stats:
    for tool_name, stats in caller_stats.items():
        line = f"    {tool_name}: {stats['pass_count']} PASS"
        print(f"\u2551  {line:<56}\u2551")
else:
    print(f"\u2551    (no variant stats available){' '*26}\u2551")
print("\u2560" + "\u2550"*58 + "\u2563")
print(f"\u2551  Output:    {output_dir:<45}\u2551")
if sarek_outputs.get("multiqc"):
    print(f"\u2551  MultiQC:   \u2705{' '*55}\u2551")
annotation_status = (
    "triggered" if trigger_annotation and nf_exit_code == 0 and qc_passed
    else "skipped (QC failed)" if not qc_passed
    else "disabled"
)
print(f"\u2551  Annotation: {annotation_status:<44}\u2551")
print("\u2560" + "\u2550"*58 + "\u2563")
print(f"\u2551  Delta tables:{' '*44}\u2551")
print(f"\u2551    {AUDIT_TABLE:<54}\u2551")
if nf_exit_code == 0 and sarek_outputs["vcfs"]:
    print(f"\u2551    {VARIANT_TABLE:<54}\u2551")
    print(f"\u2551    {SUMMARY_TABLE:<54}\u2551")
if trace_df is not None:
    print(f"\u2551    {TRACE_TABLE:<54}\u2551")
print("\u255a" + "\u2550"*58 + "\u255d")

if qc_reasons:
    print("\n\u26a0\ufe0f  QC Issues:")
    for reason in qc_reasons:
        print(f"   \u2022 {reason}")

---
## Customization Guide

### Key Differences Across Sandwich Wrappers

| Aspect | scRNA-seq | Bulk RNA-seq | Sarek (Variant Calling) |
|---|---|---|---|
| **Pipeline** | nf-core/scrnaseq v2.7.1 | nf-core/rnaseq v3.16.1 | nf-core/sarek v3.5.1 |
| **Samplesheet** | sample,fastq_1,fastq_2 | sample,fastq_1,fastq_2,strandedness | patient,sample,lane,fastq_1,fastq_2,status |
| **Key column** | \u2014 | strandedness | status (0=normal, 1=tumor) |
| **Process selectors** | STARsolo, Salmon, kallisto | STAR_ALIGN, SALMON_QUANT, RSEM | MUTECT2, MUSE, BWA_MEM, HAPLOTYPECALLER |
| **QC gates** | min_cells, median_genes, mito% | mapping_rate, detected_genes, rRNA% | variant_count, Ti/Tv, tumor_depth, concordance |
| **Downstream trigger** | Scanpy analysis notebook | Differential expression | VEP/ANNOVAR v2 annotation |
| **Output format** | h5ad (AnnData) | Gene count matrices (TSV) | VCF files + Delta tables |

### Adding New Variant Callers

nf-core/sarek supports additional tools via the `--tools` parameter. Simply add the tool name to the comma-separated list:

- `--tools mutect2,muse,strelka,freebayes` (multi-caller somatic)
- `--tools haplotypecaller,deepvariant` (germline)
- `--tools mutect2,muse,strelka,manta,ascat` (somatic SNV + SV + CNV)

The config overlay already includes process selectors for all supported tools.

### WES vs WGS

For whole-exome sequencing, add to `extra_args`:
```
--wes --intervals /path/to/targets.bed
```

### Panel of Normals (Mutect2)

For improved somatic calling with Mutect2, provide a panel of normals:
```
--pon /path/to/pon.vcf.gz
```

### Resuming Failed Runs

nf-core/sarek supports resuming from any step. Change the `step` widget:
1. `mapping` \u2192 full pipeline from FASTQ
2. `markduplicates` \u2192 resume after alignment
3. `recalibrate` \u2192 resume after MarkDuplicates
4. `variant_calling` \u2192 use pre-processed BAMs
5. `annotate` \u2192 annotate existing VCFs

Alternatively, add `-resume` to `extra_args` to use Nextflow's built-in caching.

### Version Upgrade

Same as other wrappers \u2014 just change the `pipeline_version` widget. The config overlay is version-agnostic (process names are stable across Sarek releases).

---
## \U0001f9ea Smoke Test — Synthetic Data

Validates the pre-flight layer (samplesheet building, config generation) using synthetic tumor/normal FASTQ pairs stored in UC Volumes. Does NOT run Nextflow.

In [0]:
# ── Smoke test using synthetic FASTQ data ──
# Self-contained: defines functions inline so it runs without prior cells.
# Tests the pre-flight layer independently of Nextflow.

import os, glob, re, csv, tempfile
from datetime import datetime, timezone

# ── Function definitions (mirrors cells 5 & 6) ──────────────────────────────

def build_sarek_samplesheet(fastq_dir: str, bam_dir: str, output_path: str,
                            analysis_type: str) -> dict:
    rows = []

    def classify_status(filename, analysis_type):
        if analysis_type == "germline":
            return 0
        fname = os.path.basename(filename).upper()
        tumor_patterns = ["_T_", "_TUMOR", "-T-", "-TUMOR", ".TUMOR", "_T."]
        if any(p in fname for p in tumor_patterns):
            return 1
        normal_patterns = ["_N_", "_NORMAL", "-N-", "-NORMAL", ".NORMAL", "_N."]
        if any(p in fname for p in normal_patterns):
            return 0
        return 1 if analysis_type == "somatic" else 0

    def derive_patient(filename):
        base = os.path.basename(filename)
        # Remove extension and read indicator
        base = re.sub(r'(_R[12])?(\.[^.]+)*$', '', base)
        # Remove lane suffix first
        base = re.sub(r'[_\-\.]L\d+', '', base)
        # Remove standalone T/N/tumor/normal indicators
        # Key fix: require a delimiter BEFORE the indicator (prevents
        # matching inside words like PATIENT, SENTINEL, etc.)
        patient = re.sub(r'[_\-\.](T|N|tumor|normal)(?=[_\-\.]|$)', '', base, flags=re.IGNORECASE)
        # Clean up trailing/multiple underscores
        patient = re.sub(r'[_\-\.]+$', '', patient)
        patient = re.sub(r'[_\-\.]{2,}', '_', patient)
        return patient if patient else "patient_001"

    def derive_sample(filename):
        base = os.path.basename(filename)
        sample = re.sub(r'(_R[12])?(\.[^.]+)*$', '', base)
        sample = re.sub(r'[_\-\.]L\d+', '', sample)
        return sample if sample else "sample_001"

    def extract_lane(filename):
        match = re.search(r'[_\-\.]L(\d+)', os.path.basename(filename))
        return f"L{match.group(1).zfill(3)}" if match else "L001"

    if fastq_dir and os.path.isdir(fastq_dir):
        fq_files = sorted(
            glob.glob(os.path.join(fastq_dir, "**/*.fastq.gz"), recursive=True) +
            glob.glob(os.path.join(fastq_dir, "**/*.fq.gz"), recursive=True)
        )
        r1_files = [f for f in fq_files if re.search(r'_R1[_.]', os.path.basename(f))]
        for r1 in r1_files:
            r2 = re.sub(r'_R1([_.])', r'_R2\1', r1)
            if not os.path.exists(r2):
                r2 = ""
            rows.append({
                "patient": derive_patient(r1),
                "sample": derive_sample(r1),
                "lane": extract_lane(r1),
                "fastq_1": r1,
                "fastq_2": r2,
                "status": classify_status(r1, analysis_type)
            })
        # Handle unpaired
        paired_files = set()
        for r1 in r1_files:
            paired_files.add(r1)
            r2 = re.sub(r'_R1([_.])', r'_R2\1', r1)
            if os.path.exists(r2):
                paired_files.add(r2)
        unpaired = [f for f in fq_files if f not in paired_files
                    and not re.search(r'_R2[_.]', os.path.basename(f))]
        for f in unpaired:
            rows.append({"patient": derive_patient(f), "sample": derive_sample(f),
                         "lane": extract_lane(f), "fastq_1": f, "fastq_2": "",
                         "status": classify_status(f, analysis_type)})
        header = ["patient", "sample", "lane", "fastq_1", "fastq_2", "status"]
    else:
        header = ["patient", "sample", "lane", "fastq_1", "fastq_2", "status"]

    with open(output_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=header)
        writer.writeheader()
        writer.writerows(rows)

    n_tumor = sum(1 for r in rows if r["status"] == 1)
    n_normal = sum(1 for r in rows if r["status"] == 0)
    patients = set(r["patient"] for r in rows)
    return {"n_patients": len(patients), "n_samples": len(rows),
            "n_tumor": n_tumor, "n_normal": n_normal, "input_mode": "fastq"}


def generate_eisai_sarek_config(output_dir, tools):
    config_content = """\
// Genesis Workbench — Eisai institutional config for nf-core/sarek
// Generated: {timestamp}
// Tools: {tools}

profiles {{
    conda {{
        conda.enabled = true
        docker.enabled = false
        singularity.enabled = false
    }}
}}

process {{
    errorStrategy = 'retry'
    maxRetries = 2

    withName: 'BWAMEM2_MEM' {{ cpus = 16; memory = '32.GB'; time = '24.h' }}
    withName: 'MUTECT2' {{ cpus = 4; memory = '16.GB'; time = '12.h' }}
    withName: 'MUSE_CALL' {{ cpus = 4; memory = '8.GB'; time = '12.h' }}
    withName: 'MUSE_SUMP' {{ cpus = 4; memory = '8.GB'; time = '4.h' }}
    withName: 'STRELKA_SOMATIC' {{ cpus = 8; memory = '16.GB'; time = '8.h' }}
    withName: 'HAPLOTYPECALLER' {{ cpus = 4; memory = '16.GB'; time = '12.h' }}
    withName: 'DEEPVARIANT' {{ cpus = 8; memory = '32.GB'; time = '12.h' }}
    withName: 'MANTA_SOMATIC' {{ cpus = 8; memory = '16.GB'; time = '8.h' }}
    withName: 'FREEBAYES' {{ cpus = 4; memory = '16.GB'; time = '12.h' }}
    withName: 'ASCAT' {{ cpus = 2; memory = '8.GB'; time = '8.h' }}
    withName: 'CONTROLFREEC' {{ cpus = 8; memory = '16.GB'; time = '12.h' }}
    withName: 'VEP' {{ cpus = 4; memory = '8.GB'; time = '4.h' }}
    withName: 'MULTIQC' {{ cpus = 2; memory = '8.GB'; time = '2.h' }}
    withName: 'SAMTOOLS_.*' {{ cpus = 4; memory = '8.GB'; time = '4.h' }}
}}
""".format(timestamp=datetime.now(timezone.utc).isoformat(), tools=tools)

    config_path = os.path.join(output_dir, "eisai_sarek.config")
    with open(config_path, "w") as f:
        f.write(config_content)
    return config_path


# ═══════════════════════════════════════════════════════════════════════════════
#   SMOKE TEST
# ═══════════════════════════════════════════════════════════════════════════════

TEST_FASTQ_DIR = "/Volumes/dhbl_discovery_us_dev/genesis_schema/bionemo_data/sarek_test_data"
TEST_OUTPUT_DIR = tempfile.mkdtemp(prefix="sarek_smoke_")

print("\u2550" * 60)
print("  SMOKE TEST: Pre-flight Layer Validation")
print("\u2550" * 60)

# 1. Validate input detection
fq_files = glob.glob(os.path.join(TEST_FASTQ_DIR, "**/*.fastq.gz"), recursive=True)
assert len(fq_files) == 8, f"Expected 8 FASTQ files, found {len(fq_files)}"
print(f"\n\u2705 Input detection: {len(fq_files)} FASTQ files found")

# 2. Test samplesheet building
test_samplesheet = os.path.join(TEST_OUTPUT_DIR, "test_samplesheet.csv")
stats = build_sarek_samplesheet(TEST_FASTQ_DIR, "", test_samplesheet, "somatic")

assert stats["n_patients"] == 2, f"Expected 2 patients, got {stats['n_patients']}"
assert stats["n_tumor"] >= 2, f"Expected >=2 tumor samples, got {stats['n_tumor']}"
assert stats["n_normal"] >= 2, f"Expected >=2 normal samples, got {stats['n_normal']}"
print(f"\u2705 Samplesheet: {stats['n_patients']} patients, "
      f"{stats['n_tumor']} tumor, {stats['n_normal']} normal")

# Print samplesheet contents
with open(test_samplesheet) as f:
    print(f"\n\u2500\u2500 Samplesheet contents \u2500\u2500")
    print(f.read())

# 3. Verify patient ID derivation (regression test for fixed regex)
assert stats["n_patients"] == 2, "Patient grouping failed"
with open(test_samplesheet) as f:
    reader = csv.DictReader(f)
    patients = set(row["patient"] for row in reader)
for p in patients:
    assert "PATIENT" in p, f"Patient ID '{p}' is mangled — 'PATIENT' should be preserved"
print(f"\u2705 Patient IDs: {sorted(patients)} (regex correctly preserves word boundaries)")

# 4. Test config generation
test_config = generate_eisai_sarek_config(TEST_OUTPUT_DIR, "mutect2,muse,strelka,vep")
assert os.path.exists(test_config), "Config file not created"
with open(test_config) as f:
    cfg = f.read()
assert "MUSE_CALL" in cfg, "MuSE process selector missing"
assert "MUTECT2" in cfg, "Mutect2 process selector missing"
assert "errorStrategy = 'retry'" in cfg, "Error strategy missing"
print(f"\u2705 Config: {os.path.basename(test_config)} generated with all process selectors")

print(f"\n{'\u2550' * 60}")
print(f"  \u2705 ALL SMOKE TESTS PASSED")
print(f"{'\u2550' * 60}")
print(f"\n\U0001f4cb To run full pipeline:")
print(f"   1. Set fastq_dir = '{TEST_FASTQ_DIR}'")
print(f"   2. Set output_dir = '/Volumes/dhbl_discovery_us_dev/genesis_schema/bionemo_data/sarek_test_output'")
print(f"   3. Use classic compute with Nextflow installed (not serverless)")
print(f"   4. Run all cells")